In [19]:
import pandas as pd
from wordcloud import WordCloud
import numpy as np
import tensorflow as tf
from sklearn.cluster import KMeans
from scipy.spatial.distance import cdist
from tensorflow.keras.layers import Input, Dense, Layer
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, davies_bouldin_score
import matplotlib.pyplot as plt
import seaborn as sns
import umap

In [2]:
tag_embeddings = pd.read_parquet("input/tag_embeddings.parquet").values
tag_embeddings

array([[ 2.91517749e-03, -1.21673872e-03, -3.96054471e-03, ...,
        -5.74002136e-03, -2.47626612e-03, -7.76558882e-05],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00]])

In [3]:
embedding_dim = tag_embeddings.shape[1]  # จำนวนมิติของ embedding
latent_dim = 10  # ลดให้เหลือ 10 มิติ

input_layer = Input(shape=(embedding_dim,))
encoded = Dense(128, activation='relu')(input_layer)
encoded = Dense(64, activation='relu')(encoded)
encoded = Dense(latent_dim, activation='relu')(encoded)

decoded = Dense(64, activation='relu')(encoded)
decoded = Dense(128, activation='relu')(decoded)
decoded = Dense(embedding_dim, activation='sigmoid')(decoded)

autoencoder = Model(input_layer, decoded)
encoder = Model(input_layer, encoded)

In [ ]:
autoencoder.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

In [ ]:
autoencoder.fit(tag_embeddings, tag_embeddings, epochs=50, batch_size=256, shuffle=True)

Epoch 1/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2491  
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.2437
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 985us/step - loss: 0.2313
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 967us/step - loss: 0.2045
Epoch 5/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.1516 
Epoch 6/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0729 
Epoch 7/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0148 
Epoch 8/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.0013 
Epoch 9/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 989us/step - loss: 1.4445e-04
Epoch 10/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 3.4964e-05
Epoch 11/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 949us/step - loss: 1.6116e-05
Epoch 12/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 1.0861e-05 
Epoch 13/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 965us/step - loss: 9.0312e-06
Epoch 14/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 8.1411e-06 
Epoch 15/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 7.5417e-06 
Ep

In [6]:
embedded_data = encoder.predict(tag_embeddings)

56/56 ━━━━━━━━━━━━━━━━━━━━ 0s 480us/step


In [8]:
n_clusters = 10  # จำนวน cluster ที่ต้องการ
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
clusters = kmeans.fit_predict(embedded_data)

In [9]:
cluster_centers = kmeans.cluster_centers_

In [17]:
def target_distribution(q):
    """ คำนวณ Soft Cluster Assignment Distribution """
    p = q ** 2 / tf.reduce_sum(q, axis=0)
    return (p / tf.reduce_sum(p, axis=1, keepdims=True))

In [20]:
class ClusteringLayer(Layer):
    """ Custom KMeans Clustering Layer """
    def __init__(self, n_clusters, cluster_centers, **kwargs):
        super(ClusteringLayer, self).__init__(**kwargs)
        self.n_clusters = n_clusters
        self.cluster_centers = tf.Variable(initial_value=cluster_centers, dtype=tf.float32, trainable=True)

    def call(self, inputs):
        q = 1.0 / (1.0 + tf.reduce_sum(tf.square(tf.expand_dims(inputs, axis=1) - self.cluster_centers), axis=2))
        q = tf.pow(q, (1.0 + 1.0) / 2.0)
        q = q / tf.reduce_sum(q, axis=1, keepdims=True)
        return q

In [21]:
class DECModel(tf.keras.Model):
    def __init__(self, encoder, n_clusters, cluster_centers):
        super(DECModel, self).__init__()
        self.encoder = encoder
        self.clustering_layer = ClusteringLayer(n_clusters, cluster_centers)

    def call(self, inputs):
        z = self.encoder(inputs)
        q = self.clustering_layer(z)
        return q

In [22]:
dec_model = DECModel(encoder, n_clusters, cluster_centers)
optimizer = Adam(learning_rate=0.001)

In [23]:
batch_size = 256
epochs = 50

In [24]:
for epoch in range(epochs):
    with tf.GradientTape() as tape:
        q = dec_model(tag_embeddings)
        p = target_distribution(q)
        loss = tf.reduce_mean(tf.keras.losses.KLDivergence()(p, q))
    
    grads = tape.gradient(loss, dec_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, dec_model.trainable_variables))
    
    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.numpy()}")
        
final_clusters = np.argmax(dec_model(tag_embeddings), axis=1)


Epoch 0, Loss: 0.00012596460874192417
Epoch 5, Loss: 0.00010018212924478576
Epoch 10, Loss: 7.279471174115315e-05
Epoch 15, Loss: 2.9107854061294347e-05
Epoch 20, Loss: 1.9724145658983616e-06
Epoch 25, Loss: 8.222630754062266e-08
Epoch 30, Loss: 6.428606411645887e-07
Epoch 35, Loss: 8.515498848282732e-07
Epoch 40, Loss: 9.810494248085888e-07
Epoch 45, Loss: 9.671281304690638e-07


In [25]:
clustered_df = pd.DataFrame({
    "Cluster": final_clusters
})

In [28]:
clustered_df

,Cluster
0,6
1,6
2,6
3,6
4,6
...,...
1761,6
1762,6
1763,6
1764,6


In [27]:
print(clustered_df.head(10))


   Cluster
0        6
1        6
2        6
3        6
4        6
5        6
6        6
7        6
8        6
9        6
